# Start here

**Scenario:** you have thirty minutes, a laptop, and no patience for a setup guide that fails on
step one. This page gets the repo running and explains how to read a vault. Nothing else.

Everything runs on one API key, and most of it runs on no key at all.

## One command

```bash
make setup
```

That installs the dependencies and writes your `.env`. If you already have an OpenRouter key in
another repo on this machine, it is copied across for you. The key is never printed, not to your
terminal and not into a notebook.

If you have no key yet, get one at [openrouter.ai/keys](https://openrouter.ai/keys) and put it in
`.env` by hand. A few dollars covers the whole course.

## Two ways to run a lesson

Every lesson that calls a model can run in either mode. `VAULT_MODE` in your `.env` picks one.

| Mode | What it does | Needs a key | Costs money |
|---|---|---|---|
| `replay` | Reads a saved response from `fixtures/` | no | no |
| `live` | Calls the real API | yes | yes, a fraction of a cent |

**Replay is the default, and it is what runs in CI.** The saved responses are real ones, recorded
from the API and committed. That means you can clone this repo with no key at all, run every
notebook end to end, and see exactly what the lesson showed.

Switch to `live` when you want to change a prompt and watch the answer change. That is the whole
point of a lesson, so do it often.

In [1]:
import os
from vault import load_env

# Notebooks run from their own folder, so this finds the repo's .env for you.
load_env()

# Never print a key. Length is enough to tell you it loaded.
key = os.getenv("OPENROUTER_API_KEY", "")
print("mode :", os.getenv("VAULT_MODE", "replay"))
print("model:", os.getenv("VAULT_MODEL", "not set"))
print("key  :", f"set, {len(key)} characters" if key else "not set")

mode : replay
model: google/gemini-2.5-flash-lite
key  : set, 73 characters


That prints whether your key loaded, not what it is. Get used to that habit. A key that reaches a
notebook output is a key that reaches GitHub, because this repo commits its outputs on purpose so
you can see expected results before you run anything.

A check enforces it. `make check` scans every committed output for key shaped text and fails.

## The numbers in this course come from the API, not from memory

Model prices, context limits and cache rules differ per model and change without notice. So no
lesson here writes one down. They are probed from the live API and read from a file.

```bash
make probe
```

In [2]:
from vault.client import provider_truth

truth = provider_truth()
for name, facts in truth["models"].items():
    ctx = facts["context_length"]
    print(f"{name:34} context {ctx:>9,}  in ${float(facts['prompt_usd_per_token']):.9f}/token")

google/gemini-2.5-flash-lite       context 1,048,576  in $0.000000100/token
openai/gpt-5-nano                  context   400,000  in $0.000000050/token
mistralai/mistral-nemo             context   131,072  in $0.000000019/token


Those are the real figures for the models this repo uses, fetched from the provider. If you swap
`VAULT_MODEL` in `.env`, run `make probe` again and every cost calculation follows.

This matters more than it sounds. A course that hardcodes one vendor's cache minimum teaches a
number that is wrong everywhere else, and the reader has no way to tell.

## How to read a vault

A vault is one topic, recorded as one video. Inside it, each notebook is a sub-module of about six
minutes, and every one of them runs the same eight beats.

| Beat | What you get |
|---|---|
| Mechanics | The exact fields and states, as a table |
| The picture | A diagram of what is about to happen |
| The cost | The formula, when there honestly is one |
| The failure | Code that runs and breaks in front of you |
| The diagnosis | Why it broke, traced to a specific mechanic |
| The fix | Code that runs and prints the improvement |
| The build | The production version, one function per cell |
| The gate | The check that stops it regressing, then questions |

**The failure is not decoration.** It runs, and it really fails. Watching something break and then
stop breaking is the fastest way to understand why the fix is shaped the way it is.

## Which vault to start with

Start with stateful agent runtimes and go in order. Later vaults assume you have written an agent
loop by hand and watched it misbehave.

If you already ship agents in production, the diagnosis and the gate in each sub-module are the
parts worth your time. Skim the rest.

In [3]:
from pathlib import Path

vaults = sorted(p for p in Path("..").iterdir()
                if p.is_dir() and p.name[:2].isdigit() and p.name != "00-setup")
for vault in vaults:
    lessons = sorted(vault.glob("[0-9][0-9]-*.ipynb"))
    print(f"{vault.name:38} {len(lessons)} sub-modules")

## When something goes wrong

| Symptom | Cause | Fix |
|---|---|---|
| `OPENROUTER_API_KEY is not set` | No key in `.env` | `make setup`, or add it by hand |
| `No fixture for this request` | A prompt changed since recording | `make record`, or set `VAULT_MODE=live` |
| `build/provider-truth.json is missing` | Never probed | `make probe` |
| A gate fails after your edit | It is doing its job | Read the message, it names the cell and the fix |

Run `make check` any time. It runs every gate, and it also plants a deliberately broken vault to
prove the gates still catch things. A check that has never failed is not known to check anything.